# CFEVER 事實查核（HW3）：標題比對 + bge-m3 + Reranker + RoBERTa

我沿用 sample code 的三階段架構，但把最弱的「檢索」換掉，也改了分類器的訓練資料：

```text
claim ─► ① 文件檢索：完整維基「標題比對」找候選 ─► bge-m3 向量 + 標題加分 排序 ─► 前 5 頁
      ─► ② 證據檢索：這 5 頁的所有句子 ─► bge-m3 初篩 20 句 ─► bge-reranker 重排 ─► 前 5 句
      ─► ③ 分類：[CLS] claim [SEP] 5 句證據 [SEP] ─► chinese-roberta-wwm-ext ─► SUPPORTS / REFUTES / NEI
```

| | sample code | 我的版本 | 為什麼改 |
| :--- | :--- | :--- | :--- |
| 候選頁 | 只有 train 標註過的頁 | 完整維基的標題比對 + train 標註頁 | dev / test 的正確頁不一定在 train 出現過 |
| 頁面排序 | bert-base-chinese mean pooling | bge-m3 的 [CLS] 向量 + 標題命中加分 | 原生 BERT 沒有做過檢索訓練 |
| 候選句 | 只有 train 標註過的句子 | 取回頁面的所有句子 | 同上，test 的證據句不會剛好被 train 標過 |
| 句子排序 | 向量 cosine | 向量初篩 + cross-encoder 重排 | 同一頁的句子都在講同一件事，需要更細的比對 |
| 分類訓練 | 3,000 筆、1 epoch、檢索結果當證據 | 全部 train、2 epochs、補進標準證據 | 避免拿無關句子硬學 SUPPORTS / REFUTES |
| 報告數字 | 手動抄 | 第 7 節自動印出填好數字的報告 | 數字一定來自這一次執行 |

**執行方式**：Kaggle 和 Colab 都用這一份（第 1 節會自動判斷環境）。開 GPU 後由上往下 Run all，T4 大約 40～60 分鐘。詳細步驟寫在 `index.md`。

## 1. 環境、套件與參數

In [ ]:
# 生成式 AI 使用說明：這份 notebook 是我在 Claude（Anthropic，Claude Code）協助下改寫的，
# 方法取捨也參考過 ChatGPT、Gemini 的建議；程式由我自己閱讀、執行並核對結果。
# 參考來源：
#   課程 sample code：https://github.com/IKMLab/NTHU_Natural_Language_Processing/tree/main/2026/Assignments/Assignment3
#   CFEVER 資料集：https://github.com/IKMLab/CFEVER-data
#   檢索模型 BAAI/bge-m3：https://huggingface.co/BAAI/bge-m3
#   重排模型 BAAI/bge-reranker-v2-m3：https://huggingface.co/BAAI/bge-reranker-v2-m3
#   分類模型 hfl/chinese-roberta-wwm-ext：https://huggingface.co/hfl/chinese-roberta-wwm-ext
import gc
import json
import random
import re
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

SEED = 2026


def set_seed(seed):
    """固定亂數種子：抽樣、模型初始化、打亂順序都比較能重現。"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = device.type == 'cuda'  # GPU 上用半精度：更快、更省記憶體
print('使用裝置：', device, '| PyTorch', torch.__version__)
if device.type == 'cpu':
    print('⚠️ 沒有偵測到 GPU：完整資料會跑非常久，請在 Kaggle / Colab 開啟 GPU。')

In [ ]:
# ---- 自動判斷執行環境：Kaggle / Colab / 本機 ----
COLAB_DRIVE_DIR = Path('/content/drive/MyDrive/nlp-hw3')  # Colab：把競賽 ZIP（或 4 個 jsonl）放這裡
REQUIRED_FILES = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'wiki_pages.jsonl']

try:
    import google.colab  # noqa: F401  只有 Colab 才 import 得到
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
# 注意：新版 Colab 也有一個空的 /kaggle/input 資料夾，所以要先排除 Colab，才能用它判斷 Kaggle
IN_KAGGLE = not IN_COLAB and Path('/kaggle/input').exists()

if IN_KAGGLE:
    DATA_DIR = Path('/kaggle/input')       # 競賽資料要先在右側 Input 加入
    OUTPUT_DIR = Path('/kaggle/working')   # submission.csv 會出現在 Output
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = COLAB_DRIVE_DIR / 'output'  # 輸出存回 Drive，斷線也不會不見
    if all(any(COLAB_DRIVE_DIR.rglob(name)) for name in REQUIRED_FILES):
        DATA_DIR = COLAB_DRIVE_DIR
    else:
        # Drive 裡只有 ZIP：只解壓需要的 4 個檔到 Colab 本機硬碟（比直接讀 Drive 快）
        DATA_DIR = Path('/content/cfever_data')
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        zip_paths = sorted(COLAB_DRIVE_DIR.glob('*.zip'))
        if len(zip_paths) != 1:
            raise FileNotFoundError(f'{COLAB_DRIVE_DIR} 裡要剛好有 1 個競賽 ZIP，目前找到 {len(zip_paths)} 個')
        with zipfile.ZipFile(zip_paths[0]) as archive:
            for name in REQUIRED_FILES:
                target = DATA_DIR / name
                if target.exists():
                    continue
                members = [m for m in archive.infolist() if Path(m.filename).name == name]
                if len(members) != 1:
                    raise FileNotFoundError(f'ZIP 裡應該剛好有 1 份 {name}，目前找到 {len(members)} 份')
                print('解壓：', name)
                partial = DATA_DIR / (name + '.part')  # 先寫 .part，完整寫完才改名，避免斷線留下半個檔案
                with archive.open(members[0]) as source, partial.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
                partial.replace(target)
else:
    DATA_DIR = Path('.')
    OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_data_file(filename):
    """在 DATA_DIR（含子資料夾）找檔案；找到 0 份或多份都直接報錯，避免讀錯版本。"""
    direct = DATA_DIR / filename
    if direct.is_file():
        return direct
    matches = sorted(path for path in DATA_DIR.rglob(filename) if path.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(f'{filename} 找到 {len(matches)} 份，請檢查 DATA_DIR = {DATA_DIR}：{matches}')
    return matches[0]


TRAIN_PATH, DEV_PATH, TEST_PATH, WIKI_PATH = (find_data_file(name) for name in REQUIRED_FILES)
print('環境：', 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else '本機')
for name, path in zip(REQUIRED_FILES, (TRAIN_PATH, DEV_PATH, TEST_PATH, WIKI_PATH)):
    print(f'  {name:17s} → {path}')
print('輸出資料夾：', OUTPUT_DIR)

In [ ]:
# ---- 資料量（None = 全部）----
TRAIN_SAMPLES = None   # 全部 24,012 筆；只想先確認流程能跑完，可以改成 2000
DEV_SAMPLES = None     # 全部 3,000 筆；bonus / NEI bias 要在完整 dev 上挑，數字才穩
TEST_SAMPLES = None    # 只有試跑才改；不是 None 時不會寫出 submission.csv

# ---- 模型（作業規定：每個模型都要 < 1B 參數）----
RETRIEVER_NAME = 'BAAI/bge-m3'                   # 約 568M：頁面與句子的向量
RERANKER_NAME = 'BAAI/bge-reranker-v2-m3'        # 約 568M：句子重排（cross-encoder）
CLASSIFIER_NAME = 'hfl/chinese-roberta-wwm-ext'  # 約 102M：三分類
USE_RERANKER = True                              # False = 只用向量挑句子（較快，分數通常較低）

# ---- ① 文件檢索 ----
MIN_TITLE_LEN = 2         # 太短的標題（1 個字）比對沒有意義
MAX_TITLE_LEN = 30
MAX_TITLE_PAGES = 30      # 每個 claim 最多保留幾個標題命中頁（長標題優先）
DENSE_CANDIDATES = 30     # 每個 claim 再用向量從候選池補幾頁
PAGE_LEAD_CHARS = 200     # 頁面向量只看「頁名 + 開頭 200 字」
DOCUMENT_TOP_K = 5
TITLE_BONUS_GRID = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]

# ---- ② 證據檢索 ----
SENTENCE_CANDIDATES = 20  # 向量先留前 20 句，再交給 reranker
EVIDENCE_TOP_K = 5        # FEVER score 只看前 5 句，多給不扣分，所以固定給滿 5 句
EMBED_MAX_LENGTH = 128
EMBED_BATCH_SIZE = 64
RERANK_MAX_LENGTH = 256
RERANK_BATCH_SIZE = 64

# ---- ③ 分類器 ----
EPOCHS = 2
BATCH_SIZE = 16
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
NEI_BIAS_GRID = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
SAVE_CLASSIFIER = True

LABELS = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']  # 分類器輸出 0 / 1 / 2 的意思
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
NEI = 'NOT ENOUGH INFO'
NEI_ID = LABEL_TO_ID[NEI]
SUBMIT_LABEL = {'SUPPORTS': 'supports', 'REFUTES': 'refutes', NEI: NEI}  # 與 sample code 的提交格式相同

### 讀取資料

JSONL 一行就是一筆 JSON。`all_train_df` 保留全部 train（用來建立候選頁），`train_df` 是這次拿來訓練分類器的資料。
test 只保留 `id` 和 `claim`，整份 notebook 都不會碰 test 的答案。

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding='utf-8-sig') as file:
        return pd.read_json(file, lines=True)


all_train_df = read_jsonl(TRAIN_PATH)
dev_df = read_jsonl(DEV_PATH)
test_df = read_jsonl(TEST_PATH)[['id', 'claim']]
for df in (all_train_df, dev_df):
    df['label'] = df['label'].astype(str).str.upper()
for df in (all_train_df, dev_df, test_df):
    df['claim'] = df['claim'].astype(str)

train_df = all_train_df
if TRAIN_SAMPLES is not None:
    train_df = all_train_df.sample(n=min(TRAIN_SAMPLES, len(all_train_df)), random_state=SEED)
if DEV_SAMPLES is not None:
    dev_df = dev_df.sample(n=min(DEV_SAMPLES, len(dev_df)), random_state=SEED)
if TEST_SAMPLES is not None:
    test_df = test_df.head(TEST_SAMPLES)
# 重設索引：之後第 i 筆資料就對應第 i 個檢索結果
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('train / dev / test 筆數：', len(train_df), len(dev_df), len(test_df))
print('train 標籤分布：', dict(Counter(train_df['label'])))
print('dev 標籤分布：  ', dict(Counter(dev_df['label'])))
train_df[['id', 'claim', 'label', 'evidence']].head()

## 2. Document Retrieval：找出相關的維基頁面

標準證據的每一條是 `[annotation_id, evidence_id, 頁名, 句號]`；`evidence` 是好幾「組」，**找齊任一組**就算對。
下面先寫好幾個整份 notebook 都會用到的小工具：整理標準證據、正規化標題、計算 Recall、印 Markdown 表格。

In [ ]:
def gold_groups(evidence):
    """標準證據 → [ {(頁名, 句號), ...}, ... ]，一個 set 是一組要「一起找到」的證據。
    NEI 的證據長得像 [[[標註id, None, None, None]]]，會被略過；空組也略過（空集合.issubset 永遠是 True）。"""
    groups = []
    if not isinstance(evidence, list):
        return groups
    for group in evidence:
        pairs = set()
        for item in group if isinstance(group, list) else []:
            valid = (isinstance(item, list) and len(item) >= 4 and isinstance(item[2], str)
                     and isinstance(item[3], (int, float)) and not pd.isna(item[3]))
            if not valid:  # 一組裡只要有一條壞掉，整組都不算
                pairs = set()
                break
            pairs.add((item[2], int(item[3])))
        if pairs:
            groups.append(pairs)
    return groups


def is_verifiable(label):
    return label in ('SUPPORTS', 'REFUTES')


def normalize_text(text):
    """比對用的正規化：英文小寫、各種間隔號統一成「·」、去掉所有空白。"""
    text = re.sub(r'[·‧・•]', '·', text.lower())
    return re.sub(r'\s+', '', text)


def display_title(page_id):
    """頁名給人看 / 給模型讀的樣子（處理底線與 -LRB- -RRB-）。"""
    return page_id.replace('-LRB-', '(').replace('-RRB-', ')').replace('_', ' ')


def title_key(page_id):
    """頁名 → 會出現在 claim 裡的樣子：去掉結尾的消歧義括號，例如「水星 (行星)」→「水星」。"""
    title = re.sub(r'\s*[（(][^（）()]*[）)]\s*$', '', display_title(page_id))
    return normalize_text(title)


def doc_hit(groups, pages):
    """前 N 頁是否涵蓋「任一組」標準證據需要的「所有」頁面。"""
    pages = set(pages)
    return any({title for title, _ in group} <= pages for group in groups)


def evidence_hit(groups, evidence):
    """前 N 句（[頁名, 句號]）是否完整涵蓋任一組標準證據；頁名、句號都要對。"""
    found = {(title, int(sentence_id)) for title, sentence_id in evidence}
    return any(group <= found for group in groups)


def recall_at(df, predictions, hit_fn, k=None):
    """只算 SUPPORTS / REFUTES。回傳 (recall, 命中題數, 可驗證題數)。"""
    hits = total = 0
    for label, evidence, predicted in zip(df['label'], df['evidence'], predictions):
        if not is_verifiable(label):
            continue
        total += 1
        hits += hit_fn(gold_groups(evidence), predicted if k is None else predicted[:k])
    return hits / max(total, 1), hits, total


def pct(value):
    return f'{value * 100:.2f}%'


def md_table(df):
    """DataFrame → Markdown 表格字串（可以直接貼進 Report）。"""
    lines = ['| ' + ' | '.join(map(str, df.columns)) + ' |',
             '| ' + ' | '.join(':---:' for _ in df.columns) + ' |']
    lines += ['| ' + ' | '.join(map(str, row)) + ' |' for row in df.itertuples(index=False)]
    return '\n'.join(lines)


report_values = {}  # 一路收集報告要用的數字，第 7 節會自動填進報告

### 第 1 次讀維基：只讀頁名，建立「標題 → 頁面」索引

維基有一百多萬頁、約 1 GB，全部丟進 BERT 太花 GPU 時間。可是 claim 裡的主角名稱常常就是頁名，
所以我先只讀頁名，做成一個 dict（Python 的 dict 查詢是 O(1)，一百多萬個 key 也很快）。

In [ ]:
start_time = time.time()
title_index = defaultdict(list)  # 正規化標題 → [原始頁名, ...]（同一標題可能有好幾個消歧義頁）
wiki_page_count = 0
with WIKI_PATH.open(encoding='utf-8') as file:
    for line in file:
        if not line.strip():
            continue
        page_id = json.loads(line).get('id') or ''
        wiki_page_count += 1
        if not page_id or '消歧義' in page_id or '消歧义' in page_id:
            continue
        key = title_key(page_id)
        if MIN_TITLE_LEN <= len(key) <= MAX_TITLE_LEN:
            title_index[key].append(page_id)
title_index = dict(title_index)  # 之後只查不寫，轉回一般 dict 比較保險
print(f'維基共 {wiki_page_count:,} 頁；可比對的標題 {len(title_index):,} 個；耗時 {time.time() - start_time:.0f} 秒')
report_values['n_wiki_pages'] = f'{wiki_page_count:,}'

### 標題比對：claim 的每個子字串都去查表

例如 claim = 「周杰倫發行了專輯」，就去查「周杰」「周杰倫」「周杰倫發」…「專輯」，查得到的都是候選頁。
長的標題比較具體（「周杰倫」比「專輯」有用），所以排前面，每題最多留 `MAX_TITLE_PAGES` 頁。

In [ ]:
def match_titles(claim):
    text = normalize_text(claim)
    found = {}  # 頁名 → 命中的標題長度
    for start in range(len(text)):
        for end in range(start + MIN_TITLE_LEN, min(len(text), start + MAX_TITLE_LEN) + 1):
            for page_id in title_index.get(text[start:end], []):
                found[page_id] = max(found.get(page_id, 0), end - start)
    # 長標題優先；同長度時，沒有消歧義括號的頁排前面
    ranked = sorted(found, key=lambda page_id: (-found[page_id], '(' in display_title(page_id), page_id))
    return ranked[:MAX_TITLE_PAGES]


start_time = time.time()
train_title_matches = [match_titles(claim) for claim in train_df['claim']]
dev_title_matches = [match_titles(claim) for claim in dev_df['claim']]
test_title_matches = [match_titles(claim) for claim in test_df['claim']]
print(f'標題比對完成，耗時 {time.time() - start_time:.0f} 秒')

title_coverage = recall_at(dev_df, dev_title_matches, doc_hit)[0]
no_match = sum(not matches for matches in dev_title_matches)
print('範例 claim：', dev_df.loc[0, 'claim'])
print('標題命中：', dev_title_matches[0][:10])
print(f'dev 平均每題命中 {np.mean([len(m) for m in dev_title_matches]):.1f} 頁；完全沒命中的有 {no_match} 題')
print(f'只靠標題比對（最多 {MAX_TITLE_PAGES} 頁）就涵蓋標準證據頁的比例：{pct(title_coverage)}')
report_values.update(
    avg_title_matches=f'{np.mean([len(m) for m in dev_title_matches]):.1f}',
    no_title_match=f'{no_match} 題（{pct(no_match / max(len(dev_df), 1))}）',
    title_coverage=pct(title_coverage),
)

### 第 2 次讀維基：只載入「候選池」的頁面內容

候選池 = train 標註過的頁（sample code 原本就用的）∪ 所有 claim 的標題命中頁。
只有這些頁需要內容：開頭文字拿去算頁面向量，`lines` 拆成句子留給第 3 節。
`lines` 每一行是「句號\t句子\t（可能還有超連結）」，**一定要保留原始句號**，不能自己重新編號。

In [ ]:
def parse_lines(lines):
    sentences = []
    for line in lines.split('\n'):
        parts = line.split('\t')
        if len(parts) >= 2 and parts[0].strip().isdecimal() and parts[1].strip():
            sentences.append((int(parts[0]), parts[1].strip()))
    return sentences


train_gold_pages = {title for evidence in all_train_df['evidence']
                    for group in gold_groups(evidence) for title, _ in group}
matched_pages = {page_id for matches in (train_title_matches, dev_title_matches, test_title_matches)
                 for claim_matches in matches for page_id in claim_matches}
wanted_pages = train_gold_pages | matched_pages

start_time = time.time()
page_lead = {}       # 頁名 → 開頭文字（給頁面向量用）
page_sentences = {}  # 頁名 → [(原始句號, 句子), ...]
with WIKI_PATH.open(encoding='utf-8') as file:
    for line in file:
        if not line.strip():
            continue
        page = json.loads(line)
        if page.get('id') not in wanted_pages:
            continue
        sentences = parse_lines(page.get('lines') or '')
        text = page.get('text') or ' '.join(sentence for _, sentence in sentences)
        page_lead[page['id']] = text[:PAGE_LEAD_CHARS]
        page_sentences[page['id']] = sentences
print(f'train 標註頁 {len(train_gold_pages):,} + 標題命中頁 {len(matched_pages):,} → 候選池 {len(page_sentences):,} 頁'
      f'（耗時 {time.time() - start_time:.0f} 秒）')
print('train 標註頁在維基裡找不到的數量：', len(train_gold_pages - page_sentences.keys()))
report_values.update(pool_size=f'{len(page_sentences):,}', n_train_gold_pages=f'{len(train_gold_pages):,}')

### 用 bge-m3 把文字變成向量

sample code 用 bert-base-chinese 做 mean pooling，但原生 BERT 沒有為「找相似文字」訓練過，
向量都擠在差不多的方向（anisotropy），cosine 分不太出好壞。`BAAI/bge-m3` 是專門用對比學習訓練的檢索模型，
它訓練時就是拿 **[CLS] 位置**的向量，所以這裡照它的用法取 [CLS]（`pooling='cls'`）；`'mean'` 的寫法也保留下來對照。
向量做 L2 normalization 後長度是 1，**內積就等於 cosine similarity**。

兩個加速技巧：(1) GPU 上用 fp16；(2) 先把文字依長度排序再分批，同一批長度差不多，padding 浪費很少。

In [ ]:
def resolve_model(name):
    """Kaggle 若已把模型加進 Input（離線用），就用 Input 裡的資料夾；否則從 Hugging Face 下載（要開 Internet）。"""
    short_name = name.split('/')[-1].lower()
    if IN_KAGGLE:
        for config_path in sorted(Path('/kaggle/input').rglob('config.json')):
            if short_name in str(config_path.parent).lower():
                return str(config_path.parent)
    return name


def count_parameters(model):
    count = sum(parameter.numel() for parameter in model.parameters())
    assert count < 1_000_000_000, f'模型參數 {count:,} 超過 1B，違反作業規定'
    return count


retriever_path = resolve_model(RETRIEVER_NAME)
retriever_tokenizer = AutoTokenizer.from_pretrained(retriever_path)
retriever = AutoModel.from_pretrained(retriever_path).to(device).eval()
if USE_FP16:
    retriever.half()
retriever_params = count_parameters(retriever)
print(f'檢索模型 {RETRIEVER_NAME}：{retriever_params:,} 參數')
EMBED_DTYPE = torch.float16 if USE_FP16 else torch.float32  # 向量存在 CPU 上，fp16 省一半記憶體


@torch.no_grad()
def encode_texts(texts, pooling='cls', name=''):
    """文字清單 → [文字數, 向量維度] 的 tensor（已 L2 正規化，存在 CPU）。"""
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))  # 依長度排序，padding 最少
    vectors = torch.empty(len(texts), retriever.config.hidden_size, dtype=EMBED_DTYPE)
    for batch_no, start in enumerate(range(0, len(texts), EMBED_BATCH_SIZE)):
        index = order[start:start + EMBED_BATCH_SIZE]
        inputs = retriever_tokenizer([texts[i] for i in index], padding=True, truncation=True,
                                     max_length=EMBED_MAX_LENGTH, return_tensors='pt').to(device)
        hidden = retriever(**inputs).last_hidden_state  # [batch, token 數, 維度]
        if pooling == 'cls':
            pooled = hidden[:, 0]                       # 第 0 個 token = [CLS]（XLM-R 叫 <s>）
        else:                                           # mean pooling：只平均真正的 token，不算 padding
            mask = inputs['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        pooled = F.normalize(pooled.float(), p=2, dim=1)  # 長度變 1 → 內積 = cosine
        vectors[index] = pooled.to(EMBED_DTYPE).cpu()       # 放回「原本的順序」
        if (batch_no + 1) % 500 == 0:
            print(f'  {name} 已編碼 {min(start + EMBED_BATCH_SIZE, len(texts)):,} / {len(texts):,}')
    return vectors

In [ ]:
pool_pages = sorted(page_sentences)                         # 頁面向量的第 i 列 = pool_pages[i]
page_row = {page_id: row for row, page_id in enumerate(pool_pages)}
page_texts = [f'{display_title(page_id)}：{page_lead[page_id]}' for page_id in pool_pages]

start_time = time.time()
page_embeddings = encode_texts(page_texts, name='頁面')
train_claim_emb = encode_texts(train_df['claim'].tolist(), name='train claim')
dev_claim_emb = encode_texts(dev_df['claim'].tolist(), name='dev claim')
test_claim_emb = encode_texts(test_df['claim'].tolist(), name='test claim')
print(f'頁面向量 {tuple(page_embeddings.shape)}，耗時 {time.time() - start_time:.0f} 秒')

### 混合分數：cosine + 標題加分

每個 claim 的候選頁 = 向量最像的前 `DENSE_CANDIDATES` 頁 ∪ 標題命中頁。
最終分數 = cosine similarity + bonus × [這頁是否被標題命中]，取前 `DOCUMENT_TOP_K` 頁。
bonus = 0 等於只用向量；bonus 很大等於「標題命中的頁一定排前面」。我在 dev 上把幾個 bonus 都試一次，選 Recall 最高的。

In [ ]:
@torch.no_grad()
def page_candidates(claim_embeddings, title_matches):
    """回傳每個 claim 的候選頁：{頁面列號: (cosine, 是否標題命中)}。"""
    page_matrix = page_embeddings.to(device)
    results = []
    for start in range(0, len(claim_embeddings), 256):
        chunk = claim_embeddings[start:start + 256].to(device)
        similarities = chunk @ page_matrix.T  # [這批 claim 數, 候選池頁數]
        top = similarities.topk(k=min(DENSE_CANDIDATES, similarities.shape[1]), dim=1)
        top_values, top_rows = top.values.float().cpu().tolist(), top.indices.cpu().tolist()
        for offset in range(len(chunk)):
            claim_no = start + offset
            candidates = {row: (score, False) for score, row in zip(top_values[offset], top_rows[offset])}
            matched_rows = [page_row[page_id] for page_id in title_matches[claim_no] if page_id in page_row]
            if matched_rows:
                scores = page_embeddings[matched_rows].float() @ claim_embeddings[claim_no].float()
                for row, score in zip(matched_rows, scores.tolist()):
                    candidates[row] = (score, True)
            results.append(candidates)
    del page_matrix
    return results


def rank_pages(candidates, bonus, k=DOCUMENT_TOP_K):
    ranked = sorted(candidates.items(), key=lambda item: item[1][0] + bonus * item[1][1], reverse=True)
    return [pool_pages[row] for row, _ in ranked[:k]]


train_page_cands = page_candidates(train_claim_emb, train_title_matches)
dev_page_cands = page_candidates(dev_claim_emb, dev_title_matches)
test_page_cands = page_candidates(test_claim_emb, test_title_matches)

# 在 dev 上挑 bonus
bonus_rows = []
for bonus in TITLE_BONUS_GRID:
    recall, hits, total = recall_at(dev_df, [rank_pages(c, bonus) for c in dev_page_cands], doc_hit)
    bonus_rows.append({'title bonus': bonus, f'Doc Recall@{DOCUMENT_TOP_K}': pct(recall), '命中題數': f'{hits} / {total}',
                       '_recall': recall})
best_bonus = max(bonus_rows, key=lambda row: row['_recall'])['title bonus']  # 同分時取較小的 bonus
bonus_table = pd.DataFrame(bonus_rows).drop(columns='_recall')
print(md_table(bonus_table))
print('選用的 title bonus =', best_bonus)

# 三種做法、不同 N 的 Recall@N
doc_methods = {
    '只用標題比對（長標題優先）': dev_title_matches,
    '只用 bge-m3 向量（bonus = 0）': [rank_pages(c, 0.0, k=10) for c in dev_page_cands],
    f'混合（bonus = {best_bonus}）': [rank_pages(c, best_bonus, k=10) for c in dev_page_cands],
}
doc_rows = []
for method, predictions in doc_methods.items():
    row = {'方法': method}
    for k in (1, 3, 5, 10):
        row[f'R@{k}'] = pct(recall_at(dev_df, predictions, doc_hit, k=k)[0])
    doc_rows.append(row)
doc_table = pd.DataFrame(doc_rows)
candidate_coverage = recall_at(dev_df, [[pool_pages[row] for row in c] for c in dev_page_cands], doc_hit)[0]
print(md_table(doc_table))
print('候選頁（向量 + 標題）涵蓋率：', pct(candidate_coverage))

train_docs = [rank_pages(c, best_bonus) for c in train_page_cands]
dev_docs = [rank_pages(c, best_bonus) for c in dev_page_cands]
test_docs = [rank_pages(c, best_bonus) for c in test_page_cands]
doc_recall, doc_hits, verifiable_count = recall_at(dev_df, dev_docs, doc_hit)
print(f'Document Retrieval Recall@{DOCUMENT_TOP_K}: {doc_recall:.4f}（{doc_hits} / {verifiable_count}）')
print('範例 claim：', test_df.loc[0, 'claim'], '\n取回頁面：', test_docs[0])

title_only_r5 = recall_at(dev_df, doc_methods['只用標題比對（長標題優先）'], doc_hit, k=DOCUMENT_TOP_K)[0]
dense_only_r5 = recall_at(dev_df, doc_methods['只用 bge-m3 向量（bonus = 0）'], doc_hit, k=DOCUMENT_TOP_K)[0]
max_bonus_r5 = next(row['_recall'] for row in bonus_rows if row['title bonus'] == max(TITLE_BONUS_GRID))
report_values.update(
    retriever_params=f'{retriever_params / 1e6:.0f}M', best_bonus=best_bonus, n_dev=len(dev_df),
    n_verifiable=verifiable_count, doc_r5=pct(doc_recall), doc_hits=doc_hits, doc_table=md_table(doc_table),
    bonus_table=md_table(bonus_table), candidate_coverage=pct(candidate_coverage),
    doc_title_only=pct(title_only_r5), doc_dense_only=pct(dense_only_r5), doc_max_bonus=pct(max_bonus_r5),
    doc_recall_raw=doc_recall, doc_title_only_raw=title_only_r5, doc_dense_only_raw=dense_only_r5,
    doc_max_bonus_raw=max_bonus_r5,
)

## 3. Evidence Retrieval：從取回的頁面挑句子

規定是**只能從第 1 階段取回的頁面選句**。我把這些頁的每一句都當候選（不限 train 標註過的句子），
句子前面加上頁名：很多句子用「他」「該片」代稱主角，不加頁名模型不知道在講誰。

In [ ]:
needed_pages = sorted({page_id for docs in (train_docs, dev_docs, test_docs) for pages in docs for page_id in pages})
sentence_keys, sentence_texts = [], []  # 第 i 句的 (頁名, 原始句號) 和要給模型讀的文字
page_sentence_rows = {}                 # 頁名 → 這頁每一句在上面兩個 list 的位置
for page_id in needed_pages:
    rows = []
    for sentence_id, sentence in page_sentences.get(page_id, []):
        rows.append(len(sentence_keys))
        sentence_keys.append((page_id, sentence_id))
        sentence_texts.append(f'{display_title(page_id)}：{sentence}')
    page_sentence_rows[page_id] = rows
print(f'需要的頁面 {len(needed_pages):,} 頁，候選句 {len(sentence_texts):,} 句')

start_time = time.time()
sentence_embeddings = encode_texts(sentence_texts, name='句子')
print(f'句子向量 {tuple(sentence_embeddings.shape)}，耗時 {time.time() - start_time:.0f} 秒')
report_values['n_sentences'] = f'{len(sentence_texts):,}'

# 所有向量都算完了：釋放檢索模型的 GPU 記憶體，留給後面的模型
del retriever
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 第一關：向量初篩

每個 claim 只看「自己那 5 頁」的句子。`scores.topk` 回傳的是**候選清單內的位置**（local），
要用 `rows[local]` 換回全體句子清單的位置，才查得到正確的頁名和句號。

In [ ]:
def rank_sentences_dense(claim_embeddings, docs, top_n):
    """回傳每個 claim 最像的 top_n 句（句子的全域列號）。"""
    results = []
    for claim_no, pages in enumerate(docs):
        rows = [row for page_id in pages for row in page_sentence_rows.get(page_id, [])]
        if not rows:
            results.append([])
            continue
        scores = sentence_embeddings[rows].float() @ claim_embeddings[claim_no].float()
        best_local = scores.topk(k=min(top_n, len(rows))).indices.tolist()
        results.append([rows[local] for local in best_local])  # local（候選內位置）→ 全域列號
    return results


def rows_to_keys(rows_list, k):
    return [[sentence_keys[row] for row in rows[:k]] for rows in rows_list]


train_sentence_rows = rank_sentences_dense(train_claim_emb, train_docs, EVIDENCE_TOP_K)  # train 只需要前 5 句
dev_dense_rows = rank_sentences_dense(dev_claim_emb, dev_docs, SENTENCE_CANDIDATES)
test_dense_rows = rank_sentences_dense(test_claim_emb, test_docs, SENTENCE_CANDIDATES)
dev_evidence_dense = rows_to_keys(dev_dense_rows, EVIDENCE_TOP_K)
print('向量初篩的前 5 句：', dev_evidence_dense[0])

### 第二關：cross-encoder 重排

向量檢索是 **bi-encoder**：claim 和句子各自算向量再比，快，但兩邊沒有真正「一起讀」。
**cross-encoder**（`bge-reranker-v2-m3`）把 `claim + 句子` 一起丟進模型，每個字都能互相做 attention，比較準但比較慢，
所以只對初篩後的 20 句使用，而且只用在 dev / test（train 的證據用初篩結果就夠了）。

In [ ]:
if USE_RERANKER:
    reranker_path = resolve_model(RERANKER_NAME)
    reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_path)
    reranker = AutoModelForSequenceClassification.from_pretrained(reranker_path).to(device).eval()
    if USE_FP16:
        reranker.half()
    reranker_params = count_parameters(reranker)
    print(f'重排模型 {RERANKER_NAME}：{reranker_params:,} 參數')

    @torch.no_grad()
    def rerank(claims, candidate_rows, top_k):
        pairs = [(claim_no, row) for claim_no, rows in enumerate(candidate_rows) for row in rows]
        scores = torch.empty(len(pairs))
        order = sorted(range(len(pairs)), key=lambda i: len(claims[pairs[i][0]]) + len(sentence_texts[pairs[i][1]]))
        for batch_no, start in enumerate(range(0, len(pairs), RERANK_BATCH_SIZE)):
            index = order[start:start + RERANK_BATCH_SIZE]
            inputs = reranker_tokenizer([claims[pairs[i][0]] for i in index],
                                        [sentence_texts[pairs[i][1]] for i in index],
                                        padding=True, truncation='only_second',
                                        max_length=RERANK_MAX_LENGTH, return_tensors='pt').to(device)
            scores[index] = reranker(**inputs).logits.view(-1).float().cpu()  # 一對一個相關分數
            if (batch_no + 1) % 500 == 0:
                print(f'  已重排 {min(start + RERANK_BATCH_SIZE, len(pairs)):,} / {len(pairs):,} 對')
        by_claim = defaultdict(list)
        for (claim_no, row), score in zip(pairs, scores.tolist()):
            by_claim[claim_no].append((score, row))
        return [[row for _, row in sorted(by_claim[i], reverse=True)[:top_k]] for i in range(len(candidate_rows))]

    start_time = time.time()
    dev_final_rows = rerank(dev_df['claim'].tolist(), dev_dense_rows, EVIDENCE_TOP_K)
    test_final_rows = rerank(test_df['claim'].tolist(), test_dense_rows, EVIDENCE_TOP_K)
    print(f'重排完成，耗時 {time.time() - start_time:.0f} 秒')
    del reranker
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    reranker_params = 0
    dev_final_rows = [rows[:EVIDENCE_TOP_K] for rows in dev_dense_rows]
    test_final_rows = [rows[:EVIDENCE_TOP_K] for rows in test_dense_rows]

dev_evidence = rows_to_keys(dev_final_rows, EVIDENCE_TOP_K)
test_evidence = rows_to_keys(test_final_rows, EVIDENCE_TOP_K)
print('證據座標：', test_evidence[0])

### Evidence Retrieval Recall@5

一樣只算 SUPPORTS / REFUTES，前 5 句要**完整**涵蓋某一組標準證據（頁名、句號都要對）。
這是「先找頁、再找句」的整體結果，第 1 階段沒找到的頁，這裡也不可能找到；所以另外算「文件已經找對的題目」裡的證據命中率，
才分得出是哪一階段出問題。

In [ ]:
evidence_recall, evidence_hits, _ = recall_at(dev_df, dev_evidence, evidence_hit)
dense_recall = recall_at(dev_df, dev_evidence_dense, evidence_hit)[0]
candidate_recall = recall_at(dev_df, rows_to_keys(dev_dense_rows, SENTENCE_CANDIDATES), evidence_hit)[0]

doc_ok = evidence_ok = 0
for label, evidence, docs, keys in zip(dev_df['label'], dev_df['evidence'], dev_docs, dev_evidence):
    if is_verifiable(label) and doc_hit(gold_groups(evidence), docs):
        doc_ok += 1
        evidence_ok += evidence_hit(gold_groups(evidence), keys)
evidence_given_doc = evidence_ok / max(doc_ok, 1)

evidence_rows = [
    {'方法': 'bge-m3 向量取前 5 句', f'Evidence Recall@{EVIDENCE_TOP_K}': pct(dense_recall)},
    {'方法': f'bge-m3 向量前 {SENTENCE_CANDIDATES} 句（reranker 能到的上限）', f'Evidence Recall@{EVIDENCE_TOP_K}': pct(candidate_recall)},
]
if USE_RERANKER:
    evidence_rows.append({'方法': '向量初篩 + bge-reranker 重排取前 5 句', f'Evidence Recall@{EVIDENCE_TOP_K}': pct(evidence_recall)})
evidence_table = pd.DataFrame(evidence_rows)
print(md_table(evidence_table))
print(f'Evidence Retrieval Recall@{EVIDENCE_TOP_K}: {evidence_recall:.4f}（{evidence_hits} / {verifiable_count}）')
print(f'文件已找對的 {doc_ok} 題中，證據也找齊的比例：{pct(evidence_given_doc)}')
report_values.update(
    reranker_params=f'{reranker_params / 1e6:.0f}M' if USE_RERANKER else '未使用',
    ev_r5=pct(evidence_recall), ev_hits=evidence_hits, ev_dense_r5=pct(dense_recall),
    ev_cand=pct(candidate_recall), ev_given_doc=pct(evidence_given_doc), evidence_table=md_table(evidence_table),
    ev_recall_raw=evidence_recall, ev_dense_raw=dense_recall, ev_given_doc_raw=evidence_given_doc,
)

### 準備分類器的輸入文字，並存一份檢索快取

訓練資料怎麼給證據，是我覺得最關鍵的地方：
- **SUPPORTS / REFUTES**：先拿檢索到的 5 句；如果標準證據不在裡面，就放進一組標準證據、擠掉排名最後的句子。
- **NOT ENOUGH INFO**：沒有標準證據，直接用檢索到的 5 句。
- 最後把句子順序打亂，避免模型記住「標準證據總在第幾個位置」。

原因：FEVER score 規定 SUPPORTS / REFUTES 的證據沒找到就是 0 分，所以分類器真正要做好的是
「證據對的時候判斷正確」和「NEI 題要選 NEI」。sample code 直接拿檢索結果訓練，檢索錯時等於拿無關的句子硬學 SUPPORTS。

In [ ]:
page_sentence_dict = {}


def get_sentence(page_id, sentence_id):
    if page_id not in page_sentence_dict:
        page_sentence_dict[page_id] = dict(page_sentences.get(page_id, []))
    return page_sentence_dict[page_id].get(sentence_id)


def build_context(keys):
    """[(頁名, 句號), ...] → 「【頁名】句子 【頁名】句子 …」"""
    parts = []
    for page_id, sentence_id in keys:
        sentence = get_sentence(page_id, sentence_id)
        if sentence:
            parts.append(f'【{display_title(page_id)}】{sentence}')
    return ' '.join(parts)


rng = random.Random(SEED)
train_claims, train_contexts, train_label_ids = [], [], []
gold_injected = dropped = 0
for row, retrieved in zip(train_df.itertuples(), rows_to_keys(train_sentence_rows, EVIDENCE_TOP_K)):
    keys = list(retrieved)
    if row.label != NEI:
        usable = [group for group in gold_groups(row.evidence)
                  if all(get_sentence(page_id, sentence_id) for page_id, sentence_id in group)]
        if not usable:      # 標準證據句在維基裡找不到，這題沒辦法給正確證據，就不拿來訓練
            dropped += 1
            continue
        gold = sorted(min(usable, key=len))
        if not set(gold) <= set(keys):
            gold_injected += 1
        others = [key for key in keys if key not in gold]
        keys = gold + others[:max(0, EVIDENCE_TOP_K - len(gold))]
    rng.shuffle(keys)
    train_claims.append(row.claim)
    train_contexts.append(build_context(keys))
    train_label_ids.append(LABEL_TO_ID[row.label])

dev_contexts = [build_context(keys) for keys in dev_evidence]
test_contexts = [build_context(keys) for keys in test_evidence]
print(f'訓練資料 {len(train_contexts):,} 筆（補進標準證據 {gold_injected:,} 筆，找不到標準證據句而略過 {dropped:,} 筆）')
print('訓練範例：', train_claims[0], '\n          ', train_contexts[0][:200])
report_values.update(n_train_used=f'{len(train_contexts):,}', n_gold_injected=f'{gold_injected:,}', n_dropped=dropped)

retrieval_cache = {
    'dev_ids': dev_df['id'].tolist(), 'test_ids': test_df['id'].tolist(),
    'train_claims': train_claims, 'train_contexts': train_contexts, 'train_label_ids': train_label_ids,
    'dev_contexts': dev_contexts, 'test_contexts': test_contexts, 'dev_docs': dev_docs,
    'dev_evidence': dev_evidence, 'test_evidence': test_evidence, 'report_values': report_values,
}
(OUTPUT_DIR / 'retrieval_cache.json').write_text(json.dumps(retrieval_cache, ensure_ascii=False), encoding='utf-8')
print('檢索結果已存到', OUTPUT_DIR / 'retrieval_cache.json')

In [ ]:
# Colab 斷線救援：重新執行第 1 節、「讀取資料」、第 2 節第一格（gold_groups 等工具函式）和「用 bge-m3 把文字變成向量」那格
# （裡面有 resolve_model）之後，把下面改成 True，再從這一格往下執行，就不用重跑 30 分鐘的檢索。
# 正常 Run all 時保持 False，這一格什麼都不做。
LOAD_RETRIEVAL_CACHE = False
if LOAD_RETRIEVAL_CACHE:
    cache = json.loads((OUTPUT_DIR / 'retrieval_cache.json').read_text(encoding='utf-8'))
    assert cache['dev_ids'] == dev_df['id'].tolist() and cache['test_ids'] == test_df['id'].tolist(), \
        '資料筆數設定和快取不同，請重跑檢索'
    train_claims, train_contexts = cache['train_claims'], cache['train_contexts']
    train_label_ids, dev_contexts, test_contexts = cache['train_label_ids'], cache['dev_contexts'], cache['test_contexts']
    dev_docs, report_values = cache['dev_docs'], cache['report_values']
    dev_evidence = [[tuple(key) for key in keys] for keys in cache['dev_evidence']]
    test_evidence = [[tuple(key) for key in keys] for keys in cache['test_evidence']]
    print('已載入檢索快取，可以直接訓練分類器')

## 4. Classification：claim + 證據 → 三個標籤

輸入是一組句對 `[CLS] claim [SEP] 【頁名】句子… [SEP]`。`truncation='only_second'` 表示太長時只截證據、不截 claim。
這裡不先 padding 到固定長度，而是用 `DataCollatorWithPadding` 每一批補到「這批最長」就好（dynamic padding），比較快。

In [ ]:
classifier_path = resolve_model(CLASSIFIER_NAME)
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_path)


class ClaimDataset(Dataset):
    def __init__(self, claims, contexts, label_ids=None):
        self.encodings = classifier_tokenizer(list(claims), list(contexts), truncation='only_second',
                                              max_length=MAX_LENGTH)
        self.label_ids = label_ids

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, index):
        item = {name: values[index] for name, values in self.encodings.items()}
        if self.label_ids is not None:
            item['labels'] = self.label_ids[index]
        return item


collator = DataCollatorWithPadding(classifier_tokenizer)
train_dataset = ClaimDataset(train_claims, train_contexts, train_label_ids)
dev_dataset = ClaimDataset(dev_df['claim'], dev_contexts)
test_dataset = ClaimDataset(test_df['claim'], test_contexts)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
                          generator=torch.Generator().manual_seed(SEED))
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE * 4, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 4, shuffle=False, collate_fn=collator)
print('一個 batch 的形狀：', {name: tuple(tensor.shape) for name, tensor in next(iter(train_loader)).items()})

### 模型、optimizer 與學習率排程

`AutoModelForSequenceClassification` 在預訓練模型上接一個新的三分類層（隨機初始化，所以會看到提示訊息，正常）。
學習率先在前 10% 的步數從 0 慢慢升上去（warmup），再線性降到 0，避免一開始新分類層的大梯度把預訓練權重弄壞。

In [ ]:
set_seed(SEED)
classifier = AutoModelForSequenceClassification.from_pretrained(
    classifier_path, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL_TO_ID,
).to(device)
classifier_params = count_parameters(classifier)
print(f'分類模型 {CLASSIFIER_NAME}：{classifier_params:,} 參數')

optimizer = torch.optim.AdamW(classifier.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)  # fp16 訓練時把 loss 放大，避免梯度太小變成 0


@torch.no_grad()
def predict_logits(loader):
    classifier.eval()
    outputs = []
    for batch in loader:
        batch = {name: tensor.to(device) for name, tensor in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16):
            outputs.append(classifier(**batch).logits.float().cpu())
    return torch.cat(outputs)


def logits_to_labels(logits, nei_bias=0.0):
    adjusted = logits.clone()
    adjusted[:, NEI_ID] += nei_bias  # 第 5 節會用到：NEI 的分數加一點 / 減一點
    return [LABELS[index] for index in adjusted.argmax(dim=1).tolist()]


def label_accuracy(df, predicted_labels):
    return float(np.mean([predicted == gold for predicted, gold in zip(predicted_labels, df['label'])]))


def fever_score(df, predicted_labels, predicted_evidence):
    """標籤對，而且（NEI 以外）前 5 句完整命中一組標準證據，才算這題對。"""
    hits = 0
    for gold_label, evidence, label, keys in zip(df['label'], df['evidence'], predicted_labels, predicted_evidence):
        if label == gold_label and (gold_label == NEI or evidence_hit(gold_groups(evidence), keys[:5])):
            hits += 1
    return hits / max(len(df), 1)

### Training

每一批固定 5 個步驟：清梯度 → 前向算 loss → 反向傳播 → 梯度裁剪 → 更新參數（再讓學習率往下走一步）。
每個 epoch 結束都在 dev 上看一次 Accuracy 和 FEVER score，報告第 3 題會用到這張表。

In [ ]:
epoch_rows = []
for epoch in range(1, EPOCHS + 1):
    classifier.train()
    total_loss, start_time = 0.0, time.time()
    for step, batch in enumerate(train_loader, start=1):
        batch = {name: tensor.to(device) for name, tensor in batch.items()}
        optimizer.zero_grad()                                                    # 1. 清掉上一批的梯度
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16):
            loss = classifier(**batch).loss                                      # 2. 前向 + cross entropy
        scaler.scale(loss).backward()                                            # 3. 反向傳播算梯度
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)   # 4. 梯度裁剪
        scaler.step(optimizer)                                                   # 5. 更新參數
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        if step % 200 == 0 or step == len(train_loader):
            print(f'Epoch {epoch} step {step}/{len(train_loader)}  平均 loss = {total_loss / step:.4f}'
                  f'  （{time.time() - start_time:.0f} 秒）')
    epoch_dev_labels = logits_to_labels(predict_logits(dev_loader))
    epoch_rows.append({'epoch': epoch, 'train loss': f'{total_loss / len(train_loader):.4f}',
                       'dev Accuracy': pct(label_accuracy(dev_df, epoch_dev_labels)),
                       'dev FEVER score': pct(fever_score(dev_df, epoch_dev_labels, dev_evidence))})
    print(md_table(pd.DataFrame(epoch_rows[-1:])))

epoch_table = pd.DataFrame(epoch_rows)
if SAVE_CLASSIFIER:
    classifier.save_pretrained(OUTPUT_DIR / 'classifier')
    classifier_tokenizer.save_pretrained(OUTPUT_DIR / 'classifier')
report_values.update(classifier_params=f'{classifier_params / 1e6:.0f}M', epoch_table=md_table(epoch_table))

## 5. Validation：Accuracy、FEVER score 與 NEI bias

**Accuracy** 只比對標籤；**FEVER score** 還要求 SUPPORTS / REFUTES 的前 5 句證據完整命中（NEI 只看標籤）。
兩者看的東西不同：證據沒找到的 S/R 題，標籤猜對也拿不到 FEVER 分數。所以我在 NEI 的 logit 加上不同的 bias，
在 dev 上找 FEVER score 最高的那個，再套用到 test。

In [ ]:
dev_logits = predict_logits(dev_loader)
bias_rows = []
for bias in NEI_BIAS_GRID:
    labels_with_bias = logits_to_labels(dev_logits, bias)
    bias_rows.append({'NEI bias': bias,
                      'Accuracy': pct(label_accuracy(dev_df, labels_with_bias)),
                      'FEVER score': pct(fever_score(dev_df, labels_with_bias, dev_evidence)),
                      '預測為 NEI 的比例': pct(labels_with_bias.count(NEI) / len(labels_with_bias)),
                      '_fever': fever_score(dev_df, labels_with_bias, dev_evidence)})
best_bias = max(bias_rows, key=lambda row: (row['_fever'], -abs(row['NEI bias'])))['NEI bias']  # 同分選離 0 近的
bias_table = pd.DataFrame(bias_rows).drop(columns='_fever')
print(md_table(bias_table))
print('選用的 NEI bias =', best_bias)

dev_labels_plain = logits_to_labels(dev_logits, 0.0)
dev_labels = logits_to_labels(dev_logits, best_bias)
accuracy_plain, accuracy_best = label_accuracy(dev_df, dev_labels_plain), label_accuracy(dev_df, dev_labels)
fever_plain = fever_score(dev_df, dev_labels_plain, dev_evidence)
fever_best = fever_score(dev_df, dev_labels, dev_evidence)
print(f'Classification Accuracy: {accuracy_best:.4f}（bias = 0 時 {accuracy_plain:.4f}）')
print(f'FEVER score（本地估計）: {fever_best:.4f}（bias = 0 時 {fever_plain:.4f}）')

confusion = pd.crosstab(pd.Series(dev_df['label'], name='正確答案'), pd.Series(dev_labels, name='預測'))
confusion = confusion.reindex(index=LABELS, columns=LABELS, fill_value=0)
confusion_table = confusion.reset_index().rename(columns={'正確答案': '正確答案 \\ 預測'})
print(md_table(confusion_table))

# 三階段漏斗：可驗證題一路追下去
funnel = Counter()
for label, evidence, docs, keys, predicted in zip(dev_df['label'], dev_df['evidence'], dev_docs, dev_evidence, dev_labels):
    if not is_verifiable(label):
        continue
    groups = gold_groups(evidence)
    funnel['total'] += 1
    if doc_hit(groups, docs):
        funnel['doc'] += 1
        if evidence_hit(groups, keys):
            funnel['evidence'] += 1
            funnel['label'] += predicted == label
nei_total = int((dev_df['label'] == NEI).sum())
nei_correct = sum(predicted == NEI for label, predicted in zip(dev_df['label'], dev_labels) if label == NEI)
total = max(funnel['total'], 1)
funnel_table = pd.DataFrame([
    {'階段': 'SUPPORTS / REFUTES 題數', '題數': funnel['total'], '佔可驗證題': '100.00%'},
    {'階段': f'① 前 {DOCUMENT_TOP_K} 頁找齊證據頁', '題數': funnel['doc'], '佔可驗證題': pct(funnel['doc'] / total)},
    {'階段': f'② 前 {EVIDENCE_TOP_K} 句找齊證據句', '題數': funnel['evidence'], '佔可驗證題': pct(funnel['evidence'] / total)},
    {'階段': '③ 標籤也判斷正確（= 拿到 FEVER 分數）', '題數': funnel['label'], '佔可驗證題': pct(funnel['label'] / total)},
])
print(md_table(funnel_table))
print(f'NOT ENOUGH INFO：{nei_correct} / {nei_total} 題判斷正確')

report_values.update(
    best_bias=best_bias, acc_best=pct(accuracy_best), acc_plain=pct(accuracy_plain),
    fever_best=pct(fever_best), fever_plain=pct(fever_plain), bias_table=md_table(bias_table),
    confusion_table=md_table(confusion_table), funnel_table=md_table(funnel_table),
    nei_correct=nei_correct, nei_total=nei_total,
    funnel_drops={'文件檢索': funnel['total'] - funnel['doc'], '證據檢索': funnel['doc'] - funnel['evidence'],
                  '分類': funnel['evidence'] - funnel['label']},
    funnel_total=funnel['total'], acc_best_raw=accuracy_best, acc_plain_raw=accuracy_plain,
    fever_best_raw=fever_best, fever_plain_raw=fever_plain,
)

In [ ]:
# 錯誤分析：各挑幾題來看（寫報告第 4 題時可以拿來舉例）
def show_examples(title, condition, limit=3):
    print(f'\n===== {title} =====')
    shown = 0
    for i, row in dev_df.iterrows():
        if shown < limit and condition(i, row):
            gold_pages = sorted({page for group in gold_groups(row['evidence']) for page, _ in group})
            print(f"[{row['label']} → 預測 {dev_labels[i]}] {row['claim']}")
            print(f'   標準證據頁：{gold_pages}\n   取回頁面：  {dev_docs[i]}')
            shown += 1


show_examples('文件檢索失敗', lambda i, row: is_verifiable(row['label'])
              and not doc_hit(gold_groups(row['evidence']), dev_docs[i]))
show_examples('頁面對了、句子沒找齊', lambda i, row: is_verifiable(row['label'])
              and doc_hit(gold_groups(row['evidence']), dev_docs[i])
              and not evidence_hit(gold_groups(row['evidence']), dev_evidence[i]))
show_examples('證據找齊、標籤判斷錯', lambda i, row: is_verifiable(row['label'])
              and evidence_hit(gold_groups(row['evidence']), dev_evidence[i]) and dev_labels[i] != row['label'])

## 6. 產生 submission.csv

三欄：`id`、`predicted_label`（supports / refutes / NOT ENOUGH INFO）、`predicted_evidence`（JSON 字串，NEI 填 `[]`）。
順序和 `test.jsonl` 完全相同。

In [ ]:
test_labels = logits_to_labels(predict_logits(test_loader), best_bias)
rows = []
for claim_id, label, keys in zip(test_df['id'], test_labels, test_evidence):
    evidence = [] if label == NEI else [[page_id, int(sentence_id)] for page_id, sentence_id in keys[:EVIDENCE_TOP_K]]
    rows.append({'id': claim_id, 'predicted_label': SUBMIT_LABEL[label],
                 'predicted_evidence': json.dumps(evidence, ensure_ascii=False)})
submission = pd.DataFrame(rows)
assert len(submission) == len(test_df)
assert submission['id'].tolist() == test_df['id'].tolist()
assert submission['predicted_label'].isin(SUBMIT_LABEL.values()).all()
if TEST_SAMPLES is None:
    submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False, encoding='utf-8')
    print('已寫出', OUTPUT_DIR / 'submission.csv', f'（{len(submission)} 筆）')
else:
    print('⚠️ TEST_SAMPLES 不是 None（試跑模式），沒有寫出 submission.csv')
print('test 預測分布：', dict(Counter(test_labels)))
submission.head()

## 7. 報告自動填空

下面這格會把這次執行的數字填進報告，印出來並存成 `report_filled.md`。
**把印出來的內容整段貼到最下面的 Report 格子**（題目和格式沒有改，只填 Answer），再補上 Kaggle 分數即可。
想改措辭可以直接在貼上後修改；數字請不要手動改。

In [ ]:
v = dict(report_values)


def compare(new, old, unit='個百分點'):
    diff = (new - old) * 100
    if abs(diff) < 0.5:
        return f'幾乎一樣（差 {diff:+.2f} {unit}）'
    return f'{"提升" if diff > 0 else "下降"}了 {abs(diff):.2f} {unit}'


best_single = max(v['doc_title_only_raw'], v['doc_dense_only_raw'])
v['doc_compare_text'] = (
    f"只用標題比對的 Recall@{DOCUMENT_TOP_K} 是 {v['doc_title_only']}，只用向量是 {v['doc_dense_only']}，"
    f"兩種合起來是 {v['doc_r5']}，比單用較好的那一種{compare(v['doc_recall_raw'], best_single)}。"
)
if v['best_bonus'] == 0:
    v['bonus_text'] = '結果 bonus = 0 就是最好的，代表 bge-m3 自己已經會把標題命中的頁排在前面，額外加分沒有幫助。'
elif v['doc_max_bonus_raw'] >= v['doc_recall_raw'] - 1e-9:
    v['bonus_text'] = (f"只要給標題命中的頁一點加分（bonus = {v['best_bonus']}），Recall@{DOCUMENT_TOP_K} 就從 "
                       f"{v['doc_dense_only']} 變成 {v['doc_r5']}，再把 bonus 加大到 {max(TITLE_BONUS_GRID)} 也一樣是 "
                       f"{v['doc_max_bonus']}。這代表「claim 裡出現的頁名」是非常強的線索，標題命中的頁幾乎都該排在前面。")
else:
    v['bonus_text'] = (f"bonus = {v['best_bonus']} 時 Recall@{DOCUMENT_TOP_K} 最高（{v['doc_r5']}），"
                       f"但 bonus 開到 {max(TITLE_BONUS_GRID)} 時下降到 {v['doc_max_bonus']}。我推測是因為國名、年份這類很通用的短標題"
                       '也會被比對到，加太多分會把真正的頁擠出前 5 名，所以標題比對適合當「加分」而不是「絕對優先」。')
if USE_RERANKER:
    v['evidence_compare_text'] = (
        f"只用向量挑 5 句的 Recall 是 {v['ev_dense_r5']}，加上 reranker 之後是 {v['ev_r5']}，"
        f"{compare(v['ev_recall_raw'], v['ev_dense_raw'])}；向量前 {SENTENCE_CANDIDATES} 句的 Recall 是 {v['ev_cand']}，"
        'reranker 再厲害也不會超過這個上限。'
    )
else:
    v['evidence_compare_text'] = f"這次沒有開 reranker，只用向量挑 5 句，Recall 是 {v['ev_dense_r5']}。"
if v['ev_given_doc_raw'] >= 0.75:
    v['evidence_bottleneck_text'] = '只要頁面找對，大部分題目都能把證據句找齊，證據 Recall 的損失主要來自第 1 階段沒有找到正確頁面。'
else:
    v['evidence_bottleneck_text'] = '就算頁面已經找對，挑句子這一步還是會漏掉不少，之後可以用 train 的標準證據微調 reranker 來改善。'

drops = v['funnel_drops']
worst = max(drops, key=drops.get)
advice = {
    '文件檢索': '代表還有不少正確頁面沒被找到（例如 claim 用的是別名、簡稱，頁名沒有直接出現），這是之後最值得改進的地方。',
    '證據檢索': '代表頁面找對了但句子沒挑齊，之後可以用 train 的標準證據微調 reranker，或讓多句證據的題目多保留幾句。',
    '分類': '代表證據已經找到但分類器判斷錯，之後可以換更大的分類模型（例如 large 版）或多訓練幾個 epoch。',
}
v['funnel_text'] = (f"損失最多的是「{worst}」這一步，少了 {drops[worst]} 題（佔可驗證題 {pct(drops[worst] / max(v['funnel_total'], 1))}），"
                    + advice[worst] + f"另外，NOT ENOUGH INFO 的 {v['nei_total']} 題中有 {v['nei_correct']} 題判斷正確。")
if v['best_bias'] == 0:
    v['bias_text'] = f"結果 bias = 0 時 FEVER score 就最高（{v['fever_best']}），所以最後沒有調整，test 也用 bias = 0。"
else:
    direction = '多猜 NEI' if v['best_bias'] > 0 else '少猜 NEI'
    reason = ('SUPPORTS / REFUTES 的題目如果證據沒找到，標籤猜對也拿不到分，把沒把握的題改猜 NEI 幾乎不會損失，'
              '卻能多拿到真正 NEI 題的分數。' if v['best_bias'] > 0 else
              '模型原本太常猜 NEI，減少 NEI 之後，證據有找到的 SUPPORTS / REFUTES 題才拿得到分數。')
    v['bias_text'] = (f"bias = {v['best_bias']}（{direction}）時 FEVER score 最高，從 {v['fever_plain']} 變成 {v['fever_best']}，"
                      f"但 Accuracy 從 {v['acc_plain']} 變成 {v['acc_best']}。這說明 Accuracy 和 FEVER score 看的東西不一樣：{reason}"
                      '因為競賽是用 FEVER score 排名，所以 test 我用這個 bias。')

v.update(
    max_title_pages=MAX_TITLE_PAGES, page_lead_chars=PAGE_LEAD_CHARS, dense_candidates=DENSE_CANDIDATES,
    bonus_grid=', '.join(map(str, TITLE_BONUS_GRID)), doc_k=DOCUMENT_TOP_K, ev_k=EVIDENCE_TOP_K,
    sentence_candidates=SENTENCE_CANDIDATES, max_length=MAX_LENGTH, learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE, epochs=EPOCHS, warmup_pct=f'{WARMUP_RATIO:.0%}',
)

REPORT_TEMPLATE = r'''# Report

Your Kaggle username: zhi816

Your public leaderboard score: （提交 Kaggle 後填入）

Your private leaderboard score: （11/23 公布後填入）


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer:

我讀 sample code 時發現，它只把 train 有標註過的頁面當候選，dev / test 的正確頁面如果沒在 train 出現過，就一定找不到。所以我改成讀完整的 `wiki_pages.jsonl`（共 {n_wiki_pages} 頁），並把文件檢索拆成「先找候選、再排順序」兩步：

* **(1) 標題比對（找候選）**：維基的頁名常常就是 claim 裡的主角，例如 claim 在講某部電影，頁名通常就是那部電影的名字。我先把所有頁名正規化（去掉結尾的消歧義括號、統一「·」這類間隔號、去掉空白、英文轉小寫），做成「標題 → 頁面」的 dict，再把 claim 中所有長度 2～30 的子字串拿去查。查到的就是候選頁，長的標題比較具體所以排前面，每題最多留 {max_title_pages} 頁。這一步只是 Python 的 dict 查詢，不用 GPU、也不用斷詞。dev 平均每題命中 {avg_title_matches} 頁，完全沒有命中任何標題的有 {no_title_match}。
* **(2) 向量排序**：候選頁用 `BAAI/bge-m3`（{retriever_params} 參數）把「頁名：開頭 {page_lead_chars} 字」編成向量，claim 也編成向量，兩邊都做 L2 normalization，所以內積就是 cosine similarity。bge-m3 是專門為檢索訓練過的模型，訓練時用的是 [CLS] 位置的向量，所以我照它的設計取 [CLS]，而不是 sample code 的 mean pooling。為了照顧「claim 裡沒有直接寫出頁名」的題目，我另外在候選池（train 標註頁 {n_train_gold_pages} 頁 + 所有 claim 的標題命中頁，合計 {pool_size} 頁）裡用向量再補 {dense_candidates} 頁。
* **(3) 混合分數**：最終分數 = cosine + bonus × [這頁是否被標題命中]。bonus 我在 dev 上從 {bonus_grid} 裡挑 Recall@{doc_k} 最高的，結果是 **{best_bonus}**，取前 {doc_k} 頁進入下一階段。

**Dev 結果**：dev 共 {n_dev} 筆，Recall 只計算 SUPPORTS / REFUTES 的 {n_verifiable} 筆；某一組標準證據需要的頁面「全部」都在前 N 頁裡才算命中。

**Document Retrieval Recall@{doc_k} = {doc_r5}（{doc_hits} / {n_verifiable}）**

{doc_table}

{doc_compare_text}另外，「標題命中 + 向量補充」的候選頁涵蓋率是 {candidate_coverage}，這是排序再怎麼改都超不過的上限。

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer:

證據只能從第 1 階段取回的 {doc_k} 頁裡挑。sample code 只把 train 標註過的句子當候選，我改成把這幾頁的**每一句**都當候選（保留維基原始的句號，不重新編號），全部共編碼了 {n_sentences} 句。每一句前面我都加上頁名（「頁名：句子」），因為很多句子是用「他」「該片」來代稱主角，不加頁名的話模型不知道這句在講誰。

* **(1) 向量初篩**：用同一個 bge-m3 算 claim 和每一句的 cosine similarity，每題先留前 {sentence_candidates} 句。
* **(2) Cross-encoder 重排**：再用 `BAAI/bge-reranker-v2-m3`（{reranker_params} 參數）把「claim + 句子」一起丟進模型打分數。向量檢索是 bi-encoder，claim 和句子各自算向量再比，速度快但兩邊沒有真正一起讀；cross-encoder 讓 claim 和句子的每個字互相做 attention，比較能分辨同一頁裡內容很接近的句子。因為它比較慢，所以只用在初篩後的 {sentence_candidates} 句。
* **(3) 固定給滿 {ev_k} 句**：FEVER score 只看前 5 句，多給不會扣分，所以每題都輸出 {ev_k} 句，盡量提高命中機會。

**Dev 結果**：同樣只算 {n_verifiable} 筆可驗證的題目，頁名和句號都要對，而且要完整涵蓋某一組標準證據才算命中。

**Evidence Retrieval Recall@{ev_k} = {ev_r5}（{ev_hits} / {n_verifiable}）**

{evidence_table}

{evidence_compare_text}如果只看「文件已經找對」的題目，證據 Recall@{ev_k} 是 {ev_given_doc}：{evidence_bottleneck_text}

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer:

分類模型我用 `hfl/chinese-roberta-wwm-ext`（{classifier_params} 參數），在上面接一個三分類層（SUPPORTS / REFUTES / NOT ENOUGH INFO）。輸入是句對 `[CLS] claim [SEP] 【頁名】證據句1 【頁名】證據句2 … [SEP]`，最長 {max_length} 個 token，太長時只截斷證據、不截斷 claim。選它是因為它和 bert-base-chinese 一樣大，但用了全詞遮罩（whole word masking）和更多中文語料預訓練，在中文推論任務上通常比較好。

我花最多心思的是「訓練時證據要怎麼給」：

* **SUPPORTS / REFUTES**：先拿檢索到的 {ev_k} 句；如果標準證據不在裡面，就放進一組標準證據、擠掉排名最後的句子，最後把順序打亂。這樣模型一定看得到「真正能判斷的證據」，同時也看過和測試時一樣的雜訊句。
* **NOT ENOUGH INFO**：這類題目沒有標準證據，就直接用檢索到的 {ev_k} 句，讓模型學會「有提到主題，但資訊不夠判斷時要選 NEI」。
* 會這樣設計是因為 FEVER score 的規則：SUPPORTS / REFUTES 如果證據沒找到，標籤猜對也是 0 分，所以分類器真正要做好的是「證據對的時候判斷正確」以及「NEI 題要選 NEI」。sample code 直接拿檢索結果訓練，檢索錯的時候等於拿無關的句子硬學 SUPPORTS / REFUTES，模型學到的會是雜訊。

訓練用了 {n_train_used} 筆 train（其中 {n_gold_injected} 筆有補進標準證據；{n_dropped} 筆在維基裡找不到標準證據句，所以沒拿來訓練）。設定是 AdamW、learning rate {learning_rate}、batch size {batch_size}、{epochs} 個 epoch、前 {warmup_pct} 的步數做 warmup 之後線性下降、gradient clipping 1.0，並用 fp16 混合精度加速。每個 epoch 結束我都在 dev 上看一次：

{epoch_table}

預測時我另外在 NEI 的 logit 加上一個 bias（第 4 題說明），最後選 {best_bias}。

**Dev Accuracy = {acc_best}**（dev 全部 {n_dev} 筆，包含 NEI；分類器看的是實際檢索到的證據，這裡只比對標籤）。不加 bias 時是 {acc_plain}。混淆矩陣如下（列是正確答案、欄是預測）：

{confusion_table}

4. … Anything that can strengthen your report. (10%)

Answer:

**(1) 三階段漏斗：分數到底卡在哪一步？** 我把 dev 裡 {n_verifiable} 筆 SUPPORTS / REFUTES 一路追下去，看每一步還剩幾題：

{funnel_table}

{funnel_text}

**(2) 標題加分的消融實驗**：bonus = 0 等於只用向量排序，bonus 很大等於「標題命中的頁一定排在前面」。

{bonus_table}

{bonus_text}

**(3) NEI bias：Accuracy 和 FEVER score 的取捨**。分類器輸出 logit 後，我在 NEI 那一類加上不同的 bias，看 dev 上的變化：

{bias_table}

{bias_text}

**(4) 作業規定的檢查**：三個模型 bge-m3（{retriever_params}）、bge-reranker-v2-m3（{reranker_params}）、chinese-roberta-wwm-ext（{classifier_params}）每一個都小於 1B；全部在 Kaggle / Colab 的 GPU 上執行，沒有呼叫任何外部推理 API。訓練只用 CFEVER 的 train，dev 只用來挑 bonus、NEI bias 和報告分數，test 只讀 `id` 和 `claim`，沒有用到 test 的答案。

**(5) 限制與之後想做的事**：

* bonus 和 NEI bias 是在同一份 dev 上挑的，所以 dev 分數會比真實情況稍微樂觀一點。比較嚴謹的做法是把 dev 切成兩半，一半挑參數、另一半報告分數。
* 頁面向量只看開頭 {page_lead_chars} 字，主題在比較後面才出現的頁面會比較吃虧。
* reranker 是直接拿現成的模型來用，沒有針對 CFEVER 微調。之後可以用 train 的標準證據微調它（正例是標準證據句，負例是同一頁的其他句子），應該還能再提升證據 Recall。
* 分類器如果換成 large 版（例如 `hfl/chinese-roberta-wwm-ext-large`，約 326M），Accuracy 應該還能再高一些，但訓練時間大約是 base 的 3 倍，這次因為 GPU 時數有限先沒有做。

5. Generative AI Usage Statement

Answer:

我有使用生成式 AI，使用情況如下：

* **使用工具**：主要是 Claude（Anthropic，透過 Claude Code）；另外也把題目丟給 ChatGPT 和 Gemini 問過提升分數的方向，再由我比較後決定最後的做法。
* **使用範圍**：(1) 理解 CFEVER 三階段流程和 FEVER score 的計算方式；(2) 協助把 sample code 改寫成「標題比對 + bge-m3 + reranker + RoBERTa」的版本，以及自動把數字填進報告的程式；(3) 整理學習筆記和我可能會問的問題；(4) 報告初稿的文字整理。
* **主要 prompt**：「依照作業規範改寫 sample code 並回答 report」、「評估 Claude 和 Gemini 提出的提分建議，哪個在 GPU 時數內比較可行」、「整理我讀這份程式可能會問的問題」。
* 報告裡的數字都是我這次在 Kaggle / Colab 實際執行後印出來的結果，不是 AI 生成的；程式最上方的註解也有標註生成式 AI 的使用和參考連結。'''
report_text = REPORT_TEMPLATE.format_map(v)
(OUTPUT_DIR / 'report_filled.md').write_text(report_text, encoding='utf-8')

metrics = {key: value for key, value in v.items() if isinstance(value, (int, float, str)) and not key.endswith('_table')}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(report_text)

# Report

Your Kaggle username: zhi816

Your public leaderboard score: （提交 Kaggle 後填入）

Your private leaderboard score: （11/23 公布後填入）


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer:

我讀 sample code 時發現，它只把 train 有標註過的頁面當候選，dev / test 的正確頁面如果沒在 train 出現過，就一定找不到。所以我改成讀完整的 `wiki_pages.jsonl`（共 【待填】 頁），並把文件檢索拆成「先找候選、再排順序」兩步：

* **(1) 標題比對（找候選）**：維基的頁名常常就是 claim 裡的主角，例如 claim 在講某部電影，頁名通常就是那部電影的名字。我先把所有頁名正規化（去掉結尾的消歧義括號、統一「·」這類間隔號、去掉空白、英文轉小寫），做成「標題 → 頁面」的 dict，再把 claim 中所有長度 2～30 的子字串拿去查。查到的就是候選頁，長的標題比較具體所以排前面，每題最多留 30 頁。這一步只是 Python 的 dict 查詢，不用 GPU、也不用斷詞。dev 平均每題命中 【待填】 頁，完全沒有命中任何標題的有 【待填】。
* **(2) 向量排序**：候選頁用 `BAAI/bge-m3`（568M 參數）把「頁名：開頭 200 字」編成向量，claim 也編成向量，兩邊都做 L2 normalization，所以內積就是 cosine similarity。bge-m3 是專門為檢索訓練過的模型，訓練時用的是 [CLS] 位置的向量，所以我照它的設計取 [CLS]，而不是 sample code 的 mean pooling。為了照顧「claim 裡沒有直接寫出頁名」的題目，我另外在候選池（train 標註頁 【待填】 頁 + 所有 claim 的標題命中頁，合計 【待填】 頁）裡用向量再補 30 頁。
* **(3) 混合分數**：最終分數 = cosine + bonus × [這頁是否被標題命中]。bonus 我在 dev 上從 0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0 裡挑 Recall@5 最高的，結果是 **【待填】**，取前 5 頁進入下一階段。

**Dev 結果**：dev 共 【待填】 筆，Recall 只計算 SUPPORTS / REFUTES 的 【待填】 筆；某一組標準證據需要的頁面「全部」都在前 N 頁裡才算命中。

**Document Retrieval Recall@5 = 【待填】（【待填】 / 【待填】）**

【待填：執行第 7 節後，貼上印出的表格】

【待填：第 7 節自動產生的分析句】另外，「標題命中 + 向量補充」的候選頁涵蓋率是 【待填】，這是排序再怎麼改都超不過的上限。

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer:

證據只能從第 1 階段取回的 5 頁裡挑。sample code 只把 train 標註過的句子當候選，我改成把這幾頁的**每一句**都當候選（保留維基原始的句號，不重新編號），全部共編碼了 【待填】 句。每一句前面我都加上頁名（「頁名：句子」），因為很多句子是用「他」「該片」來代稱主角，不加頁名的話模型不知道這句在講誰。

* **(1) 向量初篩**：用同一個 bge-m3 算 claim 和每一句的 cosine similarity，每題先留前 20 句。
* **(2) Cross-encoder 重排**：再用 `BAAI/bge-reranker-v2-m3`（568M 參數）把「claim + 句子」一起丟進模型打分數。向量檢索是 bi-encoder，claim 和句子各自算向量再比，速度快但兩邊沒有真正一起讀；cross-encoder 讓 claim 和句子的每個字互相做 attention，比較能分辨同一頁裡內容很接近的句子。因為它比較慢，所以只用在初篩後的 20 句。
* **(3) 固定給滿 5 句**：FEVER score 只看前 5 句，多給不會扣分，所以每題都輸出 5 句，盡量提高命中機會。

**Dev 結果**：同樣只算 【待填】 筆可驗證的題目，頁名和句號都要對，而且要完整涵蓋某一組標準證據才算命中。

**Evidence Retrieval Recall@5 = 【待填】（【待填】 / 【待填】）**

【待填：執行第 7 節後，貼上印出的表格】

【待填：第 7 節自動產生的分析句】如果只看「文件已經找對」的題目，證據 Recall@5 是 【待填】：【待填：第 7 節自動產生的分析句】

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer:

分類模型我用 `hfl/chinese-roberta-wwm-ext`（102M 參數），在上面接一個三分類層（SUPPORTS / REFUTES / NOT ENOUGH INFO）。輸入是句對 `[CLS] claim [SEP] 【頁名】證據句1 【頁名】證據句2 … [SEP]`，最長 256 個 token，太長時只截斷證據、不截斷 claim。選它是因為它和 bert-base-chinese 一樣大，但用了全詞遮罩（whole word masking）和更多中文語料預訓練，在中文推論任務上通常比較好。

我花最多心思的是「訓練時證據要怎麼給」：

* **SUPPORTS / REFUTES**：先拿檢索到的 5 句；如果標準證據不在裡面，就放進一組標準證據、擠掉排名最後的句子，最後把順序打亂。這樣模型一定看得到「真正能判斷的證據」，同時也看過和測試時一樣的雜訊句。
* **NOT ENOUGH INFO**：這類題目沒有標準證據，就直接用檢索到的 5 句，讓模型學會「有提到主題，但資訊不夠判斷時要選 NEI」。
* 會這樣設計是因為 FEVER score 的規則：SUPPORTS / REFUTES 如果證據沒找到，標籤猜對也是 0 分，所以分類器真正要做好的是「證據對的時候判斷正確」以及「NEI 題要選 NEI」。sample code 直接拿檢索結果訓練，檢索錯的時候等於拿無關的句子硬學 SUPPORTS / REFUTES，模型學到的會是雜訊。

訓練用了 【待填】 筆 train（其中 【待填】 筆有補進標準證據；【待填】 筆在維基裡找不到標準證據句，所以沒拿來訓練）。設定是 AdamW、learning rate 2e-05、batch size 16、2 個 epoch、前 10% 的步數做 warmup 之後線性下降、gradient clipping 1.0，並用 fp16 混合精度加速。每個 epoch 結束我都在 dev 上看一次：

【待填：執行第 7 節後，貼上印出的表格】

預測時我另外在 NEI 的 logit 加上一個 bias（第 4 題說明），最後選 【待填】。

**Dev Accuracy = 【待填】**（dev 全部 【待填】 筆，包含 NEI；分類器看的是實際檢索到的證據，這裡只比對標籤）。不加 bias 時是 【待填】。混淆矩陣如下（列是正確答案、欄是預測）：

【待填：執行第 7 節後，貼上印出的表格】

4. … Anything that can strengthen your report. (10%)

Answer:

**(1) 三階段漏斗：分數到底卡在哪一步？** 我把 dev 裡 【待填】 筆 SUPPORTS / REFUTES 一路追下去，看每一步還剩幾題：

【待填：執行第 7 節後，貼上印出的表格】

【待填：第 7 節自動產生的分析句】

**(2) 標題加分的消融實驗**：bonus = 0 等於只用向量排序，bonus 很大等於「標題命中的頁一定排在前面」。

【待填：執行第 7 節後，貼上印出的表格】

【待填：第 7 節自動產生的分析句】

**(3) NEI bias：Accuracy 和 FEVER score 的取捨**。分類器輸出 logit 後，我在 NEI 那一類加上不同的 bias，看 dev 上的變化：

【待填：執行第 7 節後，貼上印出的表格】

【待填：第 7 節自動產生的分析句】

**(4) 作業規定的檢查**：三個模型 bge-m3（568M）、bge-reranker-v2-m3（568M）、chinese-roberta-wwm-ext（102M）每一個都小於 1B；全部在 Kaggle / Colab 的 GPU 上執行，沒有呼叫任何外部推理 API。訓練只用 CFEVER 的 train，dev 只用來挑 bonus、NEI bias 和報告分數，test 只讀 `id` 和 `claim`，沒有用到 test 的答案。

**(5) 限制與之後想做的事**：

* bonus 和 NEI bias 是在同一份 dev 上挑的，所以 dev 分數會比真實情況稍微樂觀一點。比較嚴謹的做法是把 dev 切成兩半，一半挑參數、另一半報告分數。
* 頁面向量只看開頭 200 字，主題在比較後面才出現的頁面會比較吃虧。
* reranker 是直接拿現成的模型來用，沒有針對 CFEVER 微調。之後可以用 train 的標準證據微調它（正例是標準證據句，負例是同一頁的其他句子），應該還能再提升證據 Recall。
* 分類器如果換成 large 版（例如 `hfl/chinese-roberta-wwm-ext-large`，約 326M），Accuracy 應該還能再高一些，但訓練時間大約是 base 的 3 倍，這次因為 GPU 時數有限先沒有做。

5. Generative AI Usage Statement

Answer:

我有使用生成式 AI，使用情況如下：

* **使用工具**：主要是 Claude（Anthropic，透過 Claude Code）；另外也把題目丟給 ChatGPT 和 Gemini 問過提升分數的方向，再由我比較後決定最後的做法。
* **使用範圍**：(1) 理解 CFEVER 三階段流程和 FEVER score 的計算方式；(2) 協助把 sample code 改寫成「標題比對 + bge-m3 + reranker + RoBERTa」的版本，以及自動把數字填進報告的程式；(3) 整理學習筆記和我可能會問的問題；(4) 報告初稿的文字整理。
* **主要 prompt**：「依照作業規範改寫 sample code 並回答 report」、「評估 Claude 和 Gemini 提出的提分建議，哪個在 GPU 時數內比較可行」、「整理我讀這份程式可能會問的問題」。
* 報告裡的數字都是我這次在 Kaggle / Colab 實際執行後印出來的結果，不是 AI 生成的；程式最上方的註解也有標註生成式 AI 的使用和參考連結。